<a href="https://colab.research.google.com/github/apsantu/ImageAI/blob/master/courier_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Python Chaincode Implementation:**

This script defines the smart contract logic for managing parcel data on the Hyperledger Fabric ledger.

In [1]:
package org.example.courier;

import org.hyperledger.fabric.contract.Context;
import org.hyperledger.fabric.contract.ContractInterface;
import org.hyperledger.fabric.contract.annotation.Contract;
import org.hyperledger.fabric.contract.annotation.Default;
import org.hyperledger.fabric.contract.annotation.Info;
import org.hyperledger.fabric.contract.annotation.Transaction;
import org.hyperledger.fabric.shim.ChaincodeException;
import org.hyperledger.fabric.shim.ChaincodeStub;
import com.owlike.genson.Genson;

@Contract(
    name = "CourierContract",
    info = @Info(
        title = "Courier Supply Chain Management",
        description = "Blockchain-based courier logistics management system",
        version = "1.0.0"
    )
)
@Default
public class CourierContract implements ContractInterface {

    private final Genson genson = new Genson();

    /**
     * Initialize the ledger with initial parcel data for testing.
     */
    @Transaction(intent = Transaction.TYPE.SUBMIT)
    public void initLedger(final Context ctx) {
        ChaincodeStub stub = ctx.getStub();
        Parcel initialParcel = new Parcel("P001", "Alice Chennai", "Bob Delhi", "Chennai Hub", "Dispatched");

        String parcelState = genson.serialize(initialParcel);
        stub.putStringState(initialParcel.getParcelID(), parcelState);
    }

    /**
     * Add new parcels to the ledger, including ID, sender, and receiver.
     */
    @Transaction(intent = Transaction.TYPE.SUBMIT)
    public Parcel createParcel(final Context ctx, final String id, final String sender, final String receiver) {
        ChaincodeStub stub = ctx.getStub();

        if (parcelExists(ctx, id)) {
            throw new ChaincodeException("Parcel " + id + " already exists");
        }

        Parcel parcel = new Parcel(id, sender, receiver, "Origin Hub", "Registered");
        String parcelState = genson.serialize(parcel);
        stub.putStringState(id, parcelState);

        return parcel;
    }

    /**
     * Update the parcel status and location on the ledger[cite: 18, 23].
     */
    @Transaction(intent = Transaction.TYPE.SUBMIT)
    public Parcel updateParcelStatus(final Context ctx, final String id, final String newLocation, final String newStatus) {
        ChaincodeStub stub = ctx.getStub();

        String parcelState = stub.getStringState(id);
        if (parcelState == null || parcelState.isEmpty()) {
            throw new ChaincodeException("Parcel " + id + " does not exist");
        }

        Parcel parcel = genson.deserialize(parcelState, Parcel.class);
        parcel.setLocation(newLocation);
        parcel.setStatus(newStatus);

        stub.putStringState(id, genson.serialize(parcel));
        return parcel;
    }

    /**
     * Query a specific parcel to provide real-time monitoring[cite: 21].
     */
    @Transaction(intent = Transaction.TYPE.EVALUATE)
    public Parcel queryParcel(final Context ctx, final String id) {
        ChaincodeStub stub = ctx.getStub();
        String parcelState = stub.getStringState(id);

        if (parcelState == null || parcelState.isEmpty()) {
            throw new ChaincodeException("Parcel " + id + " does not exist");
        }

        return genson.deserialize(parcelState, Parcel.class);
    }

    @Transaction(intent = Transaction.TYPE.EVALUATE)
    public boolean parcelExists(final Context ctx, final String id) {
        ChaincodeStub stub = ctx.getStub();
        String parcelState = stub.getStringState(id);
        return (parcelState != null && !parcelState.isEmpty());
    }
}

SyntaxError: invalid syntax (2587884219.py, line 1)

Setup & Installation:

Prerequisites:

1.   Docker & Docker Compose: For running the blockchain nodes
2.   Python 3.8: Required for the Fabric Python SDK
3.   Hyperledger Fabric Samples: To utilize the test-network for deployment







Installation Steps

1. Environment Setup:
   Clone the Hyperledger Fabric samples and navigate to the test network


In [ ]:
/*
 * Bash
 */

git clone https://github.com/hyperledger/fabric-samples.git
cd fabric-samples/test-network

In [ ]:
/*
 * Start the network
 */

./network.sh up createChannel -c courierchannel -ca

In [ ]:
/*
 * Install Dependencies
 */

* Inside your chaincode folder, create a requirements.txt:

fabric-contract-api-python==0.2.0

In [ ]:
/*
 * Deploy the chaincode
 */

./network.sh deployCC -ccn courier -ccp /path/to/python/code -ccl python

**3. Execution and Perceived Output**

**How to Execute**

Use the Peer CLI to interact with the system



*   **Add Parcel**

peer chaincode invoke -C courierchannel -n courier -c '{"Args":["create_parcel", "P2026", "Chennai Office", "Delhi Branch"]}'

*   **Update Tracking**

peer chaincode invoke -C courierchannel -n courier -c '{"Args":["update_parcel_status", "P2026", "Hyderabad Hub", "In-Transit"]}'


**Perceived Output:**

**Transparency:** All stakeholders see the same real-time location data.

**Operational Efficiency:** Automation of parcel registration and status updates significantly reduces manual errors.

**Auditability:** Every update generates a transaction record on the blockchain, creating a tamper-proof history for auditing purposes.

**Security:** Unauthorized data modification is prevented by the Hyperledger Fabric architecture.